In [1]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_ollama import OllamaLLM

Multi query retriever

In [2]:
all_docs=[
    Document(page_content="Regular walking boots heart health and can reduce depression.",metadata={"source":"H1"}),
    Document(page_content="Green leafy vegetables are rich in vitamins and minerals.",metadata={"source":"H2"}),
    Document(page_content="Swimming can improve your posture and reduce back pain.",metadata={"source":"H3"}),
    Document(page_content="Cherry red nailpolish is a popular choice for a classic look.",metadata={"source":"H4"}),
    Document(page_content="The new smartphone model has a sleek design and advanced features.",metadata={"source":"H5"})
    ]

In [3]:
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = Chroma.from_documents(documents=all_docs, embedding=embeddings)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Normal retriever

In [4]:
similarity_retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 2})


In [5]:
print(type(similarity_retriever))

<class 'langchain_core.vectorstores.base.VectorStoreRetriever'>


Multiquery retriever

In [6]:
llm = OllamaLLM(model="llama3")

In [7]:
multiquery_retriever=MultiQueryRetriever.from_llm(retriever=vectorstore.as_retriever(search_kwargs={"k": 2}), llm=llm)

In [8]:
# import requests

# # Tell the local Ollama server to download llama3 directly
# url = "http://localhost:11434/api/pull"
# data = {"name": "llama3"}

# print("Downloading llama3... This might take a few minutes.")
# response = requests.post(url, json=data, stream=True)

# for line in response.iter_lines():
#     if line:
#         print(line.decode('utf-8'), end='\r')
# print("\nDone! You can now run your retriever code.")

In [10]:
query="What can reduce depression?"
similarity_results=similarity_retriever.invoke(query)
multiquery_results=multiquery_retriever.invoke(query)
results = multiquery_results[:2]

In [11]:
for i,doc in enumerate(similarity_results):
    print(f"------ similarity result{i+1} ------")
    print(doc.page_content)
print("**"*20)
for j,doc in enumerate(multiquery_results):
    print(f"------ multiquery result{j+1} ------")
    print(doc.page_content)

------ similarity result1 ------
Regular walking boots heart health and can reduce depression.
------ similarity result2 ------
Swimming can improve your posture and reduce back pain.
****************************************
------ multiquery result1 ------
The new smartphone model has a sleek design and advanced features.
------ multiquery result2 ------
Cherry red nailpolish is a popular choice for a classic look.
------ multiquery result3 ------
Regular walking boots heart health and can reduce depression.
------ multiquery result4 ------
Swimming can improve your posture and reduce back pain.
------ multiquery result5 ------
Green leafy vegetables are rich in vitamins and minerals.


In [17]:
print(len(multiquery_results))

5


In [18]:
docs = retriever.invoke("What can reduce depression?")
print("Base Retriever:", len(docs))

multi_docs = multiquery_retriever.invoke("What can reduce depression?")
print("MQR:", len(multi_docs))

NameError: name 'retriever' is not defined